# QualityPhys - Camera Remote Vital Signs Estimator (CRVSE) Project

## Notebook P3-32: App-level battery - v3 against v2

### What this notebook does
Decides, by a rule fixed before any battery number existed, whether v3 replaces v2 as the app's default heart-rate model. v3 is NB_P3_31's arm A at seed 42 (`hr_physnet_v3`); v2 is the shipped model (`hr_physnet_v2`). Both run through the app's own estimator code and gates in four benchmark scripts; this notebook reads their outputs and applies the rule. It runs no model, reads no frame and trains nothing.

### Why
NB_P3_31 found arm A's per-window error at Fitzpatrick 5-6 well below v2's (recording level, A - v2 -3.11 bpm, 95% -4.77 to -1.46). The app does not show windows: it reports a reading or refuses. The battery asks whether that gain reaches users as readings the app reports, and whether anything else gets worse.

### Order
1. **Part 0** (section 0) - the rule, committed before any battery run.
2. **Section 1** - the runs, in PowerShell: the three regression checks, then the four benchmark scripts.
3. **Section 2** - preflight: the outputs are the runs section 1 asked for.
4. **Section 3** - the criteria and the verdict.
5. **Section 4** - description, which decides nothing.
6. **Section 5** - findings, written after the run.

## 0. The rule (Part 0) - fixed 2026-10-14, before any battery run

Agreed with Cezary on 2026-10-14. Nothing in this cell changes after the runs. A criterion that cannot be computed as written counts as not met. No run is repeated, and no threshold or code is changed, to move a result.

**Compared.** `hr_physnet_v3` against `hr_physnet_v2`, both through the app's estimator code and gates. `hr_spectral` (POS) runs beside them for reference and decides nothing.

**Definitions** (per recording unless stated)
- *eligible* - a recording whose reference yields a reading (`ref_accepted`); on ECG-Fitness, NB_P3_29 Part 0's primary set;
- *reported* - status `ok`, `degraded_capture` or `unstable`;
- *error* - the reported value minus the recording's reference heart rate (`ref_hr`; on ECG-Fitness `ref_hr_median`, the median of its ECG-referenced windows);
- *wrong* - reported, and more than 10 bpm from the reference (NB_P3_29 Part 0's threshold);
- *gain* / *loss* - an eligible recording that only v3 / only v2 reports; *net* = gains - losses;
- *like-for-like window MAE* - the window MAE on the primary-set windows both models read (each summary's `common_windows`).

**1. Primary - VitalVideos VV-60, Fitzpatrick 5-6** (23 recordings). Met when all three hold:
- coverage: net ≥ +2, with at most 1 loss;
- v3 has no more wrong readings than v2;
- on the recordings both report, v3's mean absolute error is at most 1.0 bpm above v2's.

**2. Guards - v3 not worse beyond the margin.**
- VV-60 Fitzpatrick 1-4 (37 recordings), and all 60: net ≥ -1, and no more wrong readings than v2.
- UBFC-Phys T1 (56 recordings, held out from both models): net ≥ -2, and like-for-like window MAE at most +0.5 bpm.
- v2's validation split of MCD, DLCN and UBFC-rPPG (407 recordings), through the app: like-for-like window MAE at most +0.5 bpm pooled (`shared`) and at most +0.35 bpm on MCD (NB_P3_31's guard margins), and pooled coverage (reported / eligible) at most 2 percentage points lower.

**3. Stress tests - met unless v3 does harm.** On PhysDrive (all 266 indexed recordings; no model trained on PhysDrive) and on ECG-Fitness Part 0's primary set (80 recordings), v3 has at most 2 more wrong readings than v2.

**4. Drift.** The three regression checks PASS on the code and weights the battery runs on (section 1, before the battery).

**Outcomes**
- 1, 2 and 3 all met: v3 becomes `DEFAULT_ESTIMATOR`; v2 stays registered.
- 1 not met; 2 and 3 met: v2 stays the default, v3 a registered candidate.
- Any guard or stress test not met: v2 stays the default, and the cause is found before more v3 work.

**What the result is.** A decision about the app's default model, made on 23 recordings at types 5-6. It is not evidence that v3 serves darker skin better: were v3 no better at coverage, the coverage criterion would still pass about 10-17% of the time; a properly powered test of a 12-point coverage gain needs about 75 recordings at types 5-6; and VV-60 already chose arm A over R in NB_P3_31, so this reading leans optimistic. A claim about skin tone waits for an independent set.

**Printed as description, deciding nothing.** McNemar's exact one-sided p on the gains and losses; subject-bootstrap 95% intervals (4000 draws, seed 42) for the VV-60 coverage and error differences; counts by Fitzpatrick type and site group, as aggregates only (no recording is shown beside its type or site); agreement with VitalVideos' ECG; ECG-Fitness Part 0's window-level metrics; POS on every set.

## 1. Runs

Commit this notebook before any run below: the commit dates the rule. Then, from the repository root, in this order:

```powershell
# Drift - all three must PASS (exit 0); if one does not, stop: the battery does not run
venv\Scripts\python.exe -m app.live_vitals.scripts.check_ubfc_regression --estimator hr_physnet_v2
venv\Scripts\python.exe -m app.live_vitals.scripts.check_ubfc_regression --estimator hr_physnet_v3
venv\Scripts\python.exe -m app.live_vitals.scripts.check_ubfc_regression --estimator hr_spectral

# The battery - each script runs every registered estimator: v2, v3 and hr_spectral
venv\Scripts\python.exe -m app.live_vitals.scripts.check_vitalvideos --split val --device cuda
venv\Scripts\python.exe -m app.live_vitals.scripts.check_ubfc_phys --tasks T1 --device cuda
venv\Scripts\python.exe -m app.live_vitals.scripts.check_phase3_stores --split heldout --device cuda
venv\Scripts\python.exe -m app.live_vitals.scripts.check_ecg_fitness --device cuda
```

Nothing is committed between these runs, so all four share one commit. `check_ecg_fitness` rewrites the committed v2 and POS summaries; only their `provenance` blocks should change. Then run sections 2-4.

## 2. Preflight
The outputs must be the runs section 1 asked for: one commit with no uncommitted app code, the expected weights, full runs (no `--limit`), the expected recordings, and the same recordings for every model. The cell stops otherwise.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import binomtest


def find_repo_root(start):
    '''The first folder at or above `start` holding both app/ and Data/.'''
    for folder in [start, *start.parents]:
        if (folder / 'app').is_dir() and (folder / 'Data').is_dir():
            return folder
    raise FileNotFoundError(f'no repository root at or above {start}')


REPO_ROOT = find_repo_root(Path.cwd().resolve())
DATA = REPO_ROOT / 'Data'
V2, V3, POS = 'hr_physnet_v2', 'hr_physnet_v3', 'hr_spectral'
MODELS = (V2, V3, POS)
WEIGHTS = {V2: 'f30fae250fb948e2aaece9aae5ec2d77fcc438d7ef1abbe54ae878a7eecee6b8',   # NB_P3_18, epoch 27
           V3: '3fab82a89f4863a70c18ddb3d8f8f7cbc0d923adf65f85b2d0aeaa56895e0600',   # NB_P3_31 A, seed 42, last_bn
           POS: None}
# Each battery set: output folder and the name suffix its run writes.
RUNS = {'VV-60': ('vitalvideos_eval', '_split-val'),
        'UBFC-Phys T1': ('ubfc_phys_eval', '_tasks-T1'),
        'Phase-3 held-out': ('phase3_eval', '_split-heldout'),
        'ECG-Fitness': ('ecg_fitness_eval', '')}
EXPECTED = {'VV-60': 60, 'UBFC-Phys T1': 56, 'Phase-3 held-out': 673, 'ECG-Fitness': 100,
            'shared': 407, 'PhysDrive': 266, 'ECG-Fitness primary': 80}
REPORTED = ('ok', 'degraded_capture', 'unstable')
WRONG_BPM = 10.0
SHARED = ('MCD', 'DLCN', 'UBFC-rPPG')


def load(run, model):
    '''One model's summary and recording rows for one battery set.'''
    folder, suffix = RUNS[run]
    summary = json.loads((DATA / folder / f'summary_{model}{suffix}.json').read_text(encoding='utf-8'))
    recordings = pd.read_csv(DATA / folder / f'recordings_{model}{suffix}.csv')
    return summary, recordings


OUT = {(run, m): load(run, m) for run in RUNS for m in MODELS}

problems, commits = [], set()
for (run, m), (summary, recs) in OUT.items():
    prov = summary.get('provenance') or {}
    weights = (prov.get('weights') or {}).get('sha256')
    if weights != WEIGHTS[m]:
        problems.append(f'{run} {m}: weights {weights}, expected {WEIGHTS[m]}')
    git = prov.get('git') or {}
    commits.add(git.get('commit'))
    app_changes = [f for f in git.get('changed', []) if f.startswith('app/')]
    if app_changes:
        problems.append(f'{run} {m}: run on uncommitted app code {app_changes}')
    if summary.get('limit') is not None:
        problems.append(f'{run} {m}: a --limit run')
    if len(recs) != EXPECTED[run]:
        problems.append(f'{run} {m}: {len(recs)} recordings, expected {EXPECTED[run]}')
if len(commits) != 1:
    problems.append(f'the runs come from more than one commit: {sorted(map(str, commits))}')
for run in RUNS:
    first = list(OUT[(run, V2)][1]['recording'])
    if any(list(OUT[(run, m)][1]['recording']) != first for m in MODELS):
        problems.append(f'{run}: the models did not score the same recordings')
vv_summary = OUT[('VV-60', V2)][0]
if vv_summary['split'] != 'val' or vv_summary['speed'] != 1.0:
    problems.append(f"VV-60: split {vv_summary['split']}, speed {vv_summary['speed']}")
if OUT[('UBFC-Phys T1', V2)][0]['tasks_run'] != ['T1']:
    problems.append(f"UBFC-Phys: tasks {OUT[('UBFC-Phys T1', V2)][0]['tasks_run']}")
p3 = OUT[('Phase-3 held-out', V2)][0]
for group in ('shared', 'PhysDrive'):
    if p3['split'] != 'heldout' or p3['n_recordings'].get(group) != EXPECTED[group]:
        problems.append(f"Phase-3: split {p3['split']}, {group} {p3['n_recordings'].get(group)}")
n_ecg_primary = int(OUT[('ECG-Fitness', V2)][1]['primary'].sum())
if n_ecg_primary != EXPECTED['ECG-Fitness primary']:
    problems.append(f'ECG-Fitness: {n_ecg_primary} primary recordings')

print('commit:', ', '.join(map(str, commits)))
for run, (folder, suffix) in RUNS.items():
    print(f'  {run:17} Data/{folder}/summary_<model>{suffix}.json  {len(OUT[(run, V2)][1])} recordings')
assert not problems, 'preflight failed:\n  ' + '\n  '.join(problems)
print('preflight passed: the runs, weights, code and recordings section 1 asked for')

## 3. Criteria and verdict
Every criterion of Part 0, computed from the recording rows and the like-for-like blocks, then the outcome Part 0 assigns. Drift (criterion 4) is section 1's three regression checks, run before the battery.

In [ ]:
def compare(run, ref, eligible, where=None):
    '''v3 against v2, recording by recording, on the eligible recordings of one set.'''
    v2 = OUT[(run, V2)][1].set_index('recording')
    v3 = OUT[(run, V3)][1].set_index('recording')
    rows = v2[v2[eligible].astype(bool)]
    if where is not None:
        rows = rows[where(rows)]
    ids = rows.index
    side = {}
    for name, table in ((V2, v2), (V3, v3)):
        t = table.loc[ids]
        reported = t['status'].isin(REPORTED).to_numpy()
        error = (t['value'] - t[ref]).to_numpy(dtype=float)
        side[name] = (reported, error, reported & (np.abs(error) > WRONG_BPM))
    (r2, e2, w2), (r3, e3, w3) = side[V2], side[V3]
    both = r2 & r3
    gains, losses = int((r3 & ~r2).sum()), int((r2 & ~r3).sum())
    return dict(n=len(ids), reported_v2=int(r2.sum()), reported_v3=int(r3.sum()),
                gains=gains, losses=losses, net=gains - losses,
                wrong_v2=int(w2.sum()), wrong_v3=int(w3.sum()), both=int(both.sum()),
                mae_both_v2=float(np.abs(e2[both]).mean()) if both.any() else np.nan,
                mae_both_v3=float(np.abs(e3[both]).mean()) if both.any() else np.nan,
                rows=dict(r2=r2, r3=r3, e2=e2, e3=e3))


def window_gap(run, group=None):
    '''v3 minus v2 window MAE on the primary-set windows both read, and their count.'''
    block = OUT[(run, V3)][0]['common_windows']
    m = (block[group] if group else block)[V2]['primary']
    return m['this'].get('mae', np.nan) - m['other'].get('mae', np.nan), m['n_both']


def at_most(x, limit):
    return bool(np.isfinite(x) and x <= limit + 1e-12)


def coverage(c):
    return f"gains {c['gains']}, losses {c['losses']}, net {c['net']:+d} of {c['n']}"


def wrong(c):
    return f"v2 {c['wrong_v2']}, v3 {c['wrong_v3']}"


vv_56 = compare('VV-60', 'ref_hr', 'ref_accepted', lambda d: d['fitzpatrick'].isin([5, 6]))
vv_14 = compare('VV-60', 'ref_hr', 'ref_accepted', lambda d: d['fitzpatrick'].between(1, 4))
vv_all = compare('VV-60', 'ref_hr', 'ref_accepted')
t1 = compare('UBFC-Phys T1', 'ref_hr', 'ref_accepted')
shared = compare('Phase-3 held-out', 'ref_hr', 'ref_accepted', lambda d: d['dataset'].isin(SHARED))
physdrive = compare('Phase-3 held-out', 'ref_hr', 'ref_accepted', lambda d: d['dataset'] == 'PhysDrive')
ecg = compare('ECG-Fitness', 'ref_hr_median', 'primary')
gap_t1, n_t1 = window_gap('UBFC-Phys T1', 'T1')
gap_shared, n_shared = window_gap('Phase-3 held-out', 'shared')
gap_mcd, n_mcd = window_gap('Phase-3 held-out', 'MCD')
shared_change = (shared['reported_v3'] - shared['reported_v2']) / shared['n'] if shared['n'] else np.nan
err_gap_56 = vv_56['mae_both_v3'] - vv_56['mae_both_v2']

CRITERIA = [  # (part, criterion, observed, rule, met)
    ('1 primary', 'VV-60 types 5-6: coverage', coverage(vv_56), 'net >= +2, losses <= 1',
     vv_56['net'] >= 2 and vv_56['losses'] <= 1),
    ('1 primary', 'VV-60 types 5-6: wrong readings', wrong(vv_56), 'v3 <= v2',
     vv_56['wrong_v3'] <= vv_56['wrong_v2']),
    ('1 primary', 'VV-60 types 5-6: error where both report',
     f"{vv_56['both']} recordings, v2 {vv_56['mae_both_v2']:.2f}, v3 {vv_56['mae_both_v3']:.2f} bpm",
     'v3 - v2 <= +1.0 bpm', at_most(err_gap_56, 1.0)),
    ('2 guard', 'VV-60 types 1-4: coverage', coverage(vv_14), 'net >= -1', vv_14['net'] >= -1),
    ('2 guard', 'VV-60 types 1-4: wrong readings', wrong(vv_14), 'v3 <= v2',
     vv_14['wrong_v3'] <= vv_14['wrong_v2']),
    ('2 guard', 'VV-60 all: coverage', coverage(vv_all), 'net >= -1', vv_all['net'] >= -1),
    ('2 guard', 'VV-60 all: wrong readings', wrong(vv_all), 'v3 <= v2',
     vv_all['wrong_v3'] <= vv_all['wrong_v2']),
    ('2 guard', 'UBFC-Phys T1: coverage', coverage(t1), 'net >= -2', t1['net'] >= -2),
    ('2 guard', 'UBFC-Phys T1: like-for-like window MAE', f'v3 - v2 {gap_t1:+.2f} bpm on {n_t1} windows',
     '<= +0.5 bpm', at_most(gap_t1, 0.5)),
    ('2 guard', "v2's validation, shared: like-for-like window MAE",
     f'v3 - v2 {gap_shared:+.2f} bpm on {n_shared} windows', '<= +0.5 bpm', at_most(gap_shared, 0.5)),
    ('2 guard', "v2's validation, MCD: like-for-like window MAE",
     f'v3 - v2 {gap_mcd:+.2f} bpm on {n_mcd} windows', '<= +0.35 bpm', at_most(gap_mcd, 0.35)),
    ('2 guard', "v2's validation, shared: coverage",
     f"v2 {shared['reported_v2']}, v3 {shared['reported_v3']} of {shared['n']} ({100 * shared_change:+.1f} points)",
     '>= -2 points', at_most(-shared_change, 0.02)),
    ('3 stress', 'PhysDrive: wrong readings', wrong(physdrive) + f" of {physdrive['n']}", 'v3 <= v2 + 2',
     physdrive['wrong_v3'] <= physdrive['wrong_v2'] + 2),
    ('3 stress', 'ECG-Fitness primary: wrong readings', wrong(ecg) + f" of {ecg['n']}", 'v3 <= v2 + 2',
     ecg['wrong_v3'] <= ecg['wrong_v2'] + 2),
]

print(f"{'part':9} {'criterion':50} {'observed':58} {'rule':24} result")
for part, name, observed, rule, met in CRITERIA:
    print(f"{part:9} {name:50} {observed:58} {rule:24} {'met' if met else 'NOT MET'}")

met = {p: all(c[4] for c in CRITERIA if c[0] == p) for p in ('1 primary', '2 guard', '3 stress')}
if all(met.values()):
    verdict = 'v3 becomes DEFAULT_ESTIMATOR; v2 stays registered.'
elif met['2 guard'] and met['3 stress']:
    verdict = 'v2 stays the default; v3 stays a registered candidate.'
else:
    verdict = 'v2 stays the default; the cause of the failed guard or stress test is found before more v3 work.'
print('\n' + ', '.join(f"{p}: {'met' if ok else 'NOT MET'}" for p, ok in met.items()))
print('drift: section 1 regression checks (must have passed before the battery ran)')
print(f'VERDICT: {verdict}')

## 4. Description (decides nothing)
What Part 0 lists to print beside the verdict. Counts by Fitzpatrick type and site group are aggregates; no recording appears beside its type or site.

In [ ]:
rng = np.random.default_rng(42)
N_BOOT = 4000


def mcnemar(c):
    '''Exact one-sided McNemar p for v3 reporting more than v2: the gains among the discordant.'''
    d = c['gains'] + c['losses']
    return binomtest(c['gains'], d, 0.5, alternative='greater').pvalue if d else float('nan')


def boot_interval(values):
    finite = values[np.isfinite(values)]
    return np.percentile(finite, [2.5, 97.5]) if finite.size >= 0.5 * values.size else [np.nan, np.nan]


def vv_bootstrap(c):
    '''Subject-bootstrap 95% intervals (one recording per VitalVideos subject) for the
    coverage difference and the error difference where both report.'''
    r2, r3, e2, e3 = (c['rows'][k] for k in ('r2', 'r3', 'e2', 'e3'))
    n = len(r2)
    cover, err = np.empty(N_BOOT), np.empty(N_BOOT)
    for b in range(N_BOOT):
        i = rng.integers(0, n, n)
        cover[b] = (r3[i].sum() - r2[i].sum()) / n
        both = r2[i] & r3[i]
        err[b] = np.abs(e3[i][both]).mean() - np.abs(e2[i][both]).mean() if both.any() else np.nan
    return boot_interval(cover), boot_interval(err)


print('VV-60, coverage and error differences (v3 - v2):')
for label, c in (('types 5-6', vv_56), ('types 1-4', vv_14), ('all 60', vv_all)):
    (c_lo, c_hi), (e_lo, e_hi) = vv_bootstrap(c)
    print(f"  {label:9} reported v2 {c['reported_v2']}/{c['n']}, v3 {c['reported_v3']}/{c['n']} | "
          f"coverage {100 * c['net'] / c['n']:+.1f} points (95% {100 * c_lo:+.1f} to {100 * c_hi:+.1f}) | "
          f"McNemar one-sided p {mcnemar(c):.3f} | error where both report "
          f"{c['mae_both_v3'] - c['mae_both_v2']:+.2f} bpm (95% {e_lo:+.2f} to {e_hi:+.2f})")

v2_rows = OUT[('VV-60', V2)][1]
v3_rows = OUT[('VV-60', V3)][1].set_index('recording').loc[v2_rows['recording']].reset_index()
eligible = v2_rows['ref_accepted'].astype(bool)
for field in ('fitzpatrick', 'site_group'):
    print(f'\nVV-60 by {field} (eligible recordings; reported and wrong, v2 / v3):')
    for value in sorted(v2_rows.loc[eligible, field].unique(), key=str):
        mask = eligible & (v2_rows[field] == value)
        cells = []
        for rows in (v2_rows, v3_rows):
            reported = rows.loc[mask, 'status'].isin(REPORTED)
            off = (rows.loc[mask, 'value'] - rows.loc[mask, 'ref_hr']).abs() > WRONG_BPM
            cells.append((int(reported.sum()), int((reported & off).sum())))
        print(f'  {str(value):12} n {int(mask.sum()):3d} | reported {cells[0][0]:3d} / {cells[1][0]:3d} | '
              f'wrong {cells[0][1]:2d} / {cells[1][1]:2d}')

print("\nVV-60 against the study's ECG (recordings the audit accepted, reported by both):")
ecg_ok = eligible & v2_rows['ecg_ok'].astype(bool)
both = ecg_ok & v2_rows['status'].isin(REPORTED) & v3_rows['status'].isin(REPORTED)
for name, rows in ((V2, v2_rows), (V3, v3_rows)):
    gap = (rows.loc[both, 'value'] - rows.loc[both, 'hr_ecg']).abs()
    print(f'  {name}: mean |value - ECG| {gap.mean():.2f} bpm on {int(both.sum())} recordings')

print('\nVV-60 like for like, primary set:')
gap_vv, n_vv = window_gap('VV-60')
print(f'  v3 - v2 window MAE {gap_vv:+.2f} bpm on {n_vv} windows both read')

print('\nPOS (hr_spectral), for reference - eligible, reported, wrong:')
for run, ref, elig, where, label in (
        ('VV-60', 'ref_hr', 'ref_accepted', lambda d: d['fitzpatrick'].isin([5, 6]), 'VV-60 types 5-6'),
        ('VV-60', 'ref_hr', 'ref_accepted', None, 'VV-60 all'),
        ('UBFC-Phys T1', 'ref_hr', 'ref_accepted', None, 'UBFC-Phys T1'),
        ('Phase-3 held-out', 'ref_hr', 'ref_accepted', lambda d: d['dataset'].isin(SHARED), 'shared'),
        ('Phase-3 held-out', 'ref_hr', 'ref_accepted', lambda d: d['dataset'] == 'PhysDrive', 'PhysDrive'),
        ('ECG-Fitness', 'ref_hr_median', 'primary', None, 'ECG-Fitness primary')):
    rows = OUT[(run, POS)][1]
    mask = rows[elig].astype(bool) & (where(rows) if where else True)
    reported = rows.loc[mask, 'status'].isin(REPORTED)
    off = (rows.loc[mask, 'value'] - rows.loc[mask, ref]).abs() > WRONG_BPM
    print(f'  {label:20} n {int(mask.sum()):4d} | reported {int(reported.sum()):4d} | wrong {int((reported & off).sum()):3d}')

print("\nECG-Fitness Part 0, window level by activity (confident-wrong among scored windows):")
for activity in OUT[('ECG-Fitness', V2)][0]['primary_by_activity']:
    parts = []
    for name in (V2, V3):
        g = OUT[('ECG-Fitness', name)][0]['primary_by_activity'][activity]
        parts.append(f"{name} {g['n_confident_wrong']}/{g['n_scored']} reported {g['n_reported']}/{g['n_recordings']}")
    print(f'  {activity}: ' + ' | '.join(parts))

## 5. Findings
Written after the run, from the printed output above.